# CLIP near-duplicate finder — agi-eval-data

**Free Colab T4 · Runtime → Run all · leaves tab open ~1h.**

1. Auth as your Google account (owner of the dataset folders)
2. List image files via Drive API, drop md5-exact copies → unique set
3. Fetch each as a ~448px Drive thumbnail (16 threads, logged failures)
4. Embed with open_clip ViT-B/32 fp16 — CLIP's native input is 224px crops, 448px is already generous
5. Cosine > 0.95 pair search + union-find grouping
6. Write `near-dup.csv` + contact sheet to Drive `dedup/findings/`

Checkpoints to Drive every ~2000 images — disconnect loses only the tail.

In [ ]:
# Cell 1 — auth
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
print('AUTHORIZED-AS:', drive.about().get(fields='user').execute()['user']['emailAddress'])

!pip -q install open_clip_torch
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU — Runtime > Change runtime type > T4)')

## 1 · List unique images

In [ ]:
IMG_MIMES = ('image/jpeg','image/png','image/webp','image/heic','image/heif')
FIELDS = 'nextPageToken, files(id,name,mimeType,md5Checksum)'

all_files, token = [], None
while True:
    resp = drive.files().list(q='trashed = false', pageSize=1000, fields=FIELDS,
                              supportsAllDrives=True, includeItemsFromAllDrives=True,
                              pageToken=token).execute()
    batch = [f for f in resp.get('files', []) if f.get('mimeType') in IMG_MIMES and f.get('md5Checksum')]
    all_files.extend(batch)
    print(f'...{len(all_files)} image files listed', end='\r')
    token = resp.get('nextPageToken')
    if not token:
        break
print(f'\ntotal image files: {len(all_files)}')

seen, unique = set(), []
for f in all_files:                      # keep first per md5 — exact copies stay out
    if f['md5Checksum'] in seen: continue
    seen.add(f['md5Checksum'])
    unique.append(f)
print(f'after md5 exact-dedupe: {len(unique)} unique images')
by_id = {f['id']: f for f in unique}

## 2 · Fetch Drive thumbnails (parallel, resumable)

In [ ]:
import io, os, json, time
import requests, numpy as np
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/dedup/findings'
os.makedirs(OUT_DIR, exist_ok=True)
EMB_PATH = os.path.join(OUT_DIR, 'embeddings.npy')
IDS_PATH = os.path.join(OUT_DIR, 'embedding_ids.json')

ids_all = [f['id'] for f in unique]
done = set(json.load(open(IDS_PATH))) if os.path.exists(IDS_PATH) else set()
todo = [i for i in ids_all if i not in done]
print(f'resume: {len(done)} embedded already · this session: {len(todo)}')

S = requests.Session()
def fetch(fid):
    r = S.get(f'https://lh3.googleusercontent.com/d/{fid}=w448', timeout=30)
    if r.status_code != 200:
        return fid, None, r.status_code
    try:
        return fid, Image.open(io.BytesIO(r.content)).convert('RGB'), 200
    except Exception:
        return fid, None, 'decode-error'

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

## 3 · Embed (CLIP ViT-B/32, fp16)

In [ ]:
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = model.half().cuda().eval()

embs  = list(np.load(EMB_PATH)) if os.path.exists(EMB_PATH) else []
ids_order = list(json.load(open(IDS_PATH))) if os.path.exists(IDS_PATH) else []

errors, BATCH, FETCH = [], 512, 512 * 8
with ThreadPoolExecutor(16) as pool:
    for chunk in chunks(todo, FETCH):
        results = list(pool.map(fetch, chunk))
        ok = [(fid, img) for fid, img, _ in results if img is not None]
        errors.extend((fid, st) for fid, img, st in results if img is None)
        if not ok:
            continue
        batch = torch.stack([preprocess(img) for _, img in ok]).cuda()
        with torch.no_grad():
            e = model.encode(batch.half())
            e = e / e.norm(dim=-1, keepdim=True)
        embs.extend(e.cpu().float().numpy())
        ids_order.extend(fid for fid, _ in ok)
        if len(ids_order) % 2000 < FETCH:
            np.save(EMB_PATH, np.stack(embs).astype(np.float32))
            json.dump(ids_order, open(IDS_PATH, 'w'))
            print(f'checkpoint · {len(ids_order)} embedded · {len(errors)} unreadable', end='\r')

if embs:
    np.save(EMB_PATH, np.stack(embs).astype(np.float32))
    json.dump(ids_order, open(IDS_PATH, 'w'))
print(f'\nfinal: {len(ids_order)} embedded · {len(errors)} unreadable ({len(errors)/max(len(ids_all),1)*100:.1f}%)')

## 4 · Cosine > 0.95 pairs + union-find groups

In [ ]:
from collections import Counter

THRESH = 0.95
X  = torch.from_numpy(np.stack(embs).astype(np.float32)).cuda()   # [N, D], held once
N  = X.shape[0]

# parent = union-find over global row indices 0..N-1
parent = list(range(N))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# Blocked cosine: for each block compute its [B, N] similarity against the full
# matrix (X @ X.T per block). Peak mem = B*N*4 bytes; 4096x~53k fp32 ≈ 890 MB,
# plus we EXPLICITLY keep only the upper triangle. Free T4 (16 GB) handles this.
B = 4096
for s in range(0, N, B):
    block = X[s:s+B] @ X.T                       # [B, N]
    # upper-triangle pairs with similarity >= THRESH (i>j)
    idx = torch.nonzero(block >= THRESH, as_tuple=False)
    rows, cols = idx[:, 0].tolist(), idx[:, 1].tolist()
    for bi, j in zip(rows, cols):
        i = bi + s
        if j < i:
            union(i, j)
    del block

# collapse into clusters
groups_ = {}
for i in range(N):
    groups_.setdefault(find(i), []).append(i)
clusters = sorted((g for g in groups_.values() if len(g) > 1), key=len, reverse=True)
print(f'{len(clusters)} near-dup clusters (size>1) from {N} unique images')
print('size histogram:', dict(Counter(len(c) for c in clusters)))

## 5 · Write findings to Drive + download CSV

In [ ]:
import csv, html

rows, groups_out = [], []
# ids_order[k] = Drive file id of embedding row k  ->  build global-id -> file-id lookup
gid_of = {i: ids_order[i] for i in range(len(ids_order))}

for n, cluster in enumerate(clusters, 1):
    gid = f'nd-{n:05d}'
    # keep = the member whose embedding is closest to the cluster centroid
    # (most 'average' photo). mat is [k, k] pairwise cosine inside the cluster.
    mat = X[cluster] @ X[cluster].T
    keep_local = int(mat.mean(dim=1).argmax().item())
    keep = cluster[keep_local]                     # global row index of kept image
    keep_fid = gid_of[keep]                        # its Drive file id
    for i in cluster:
        if i == keep:
            continue
        cos = float(X[i] @ X[keep])
        rows.append({'group_id': gid,
                     'kept_file_id': keep_fid,
                     'dropped_file_id': gid_of[i],
                     'cosine': f'{cos:.4f}'})
    groups_out.append({'id': gid, 'kept': keep_fid,
                       'members': [gid_of[k] for k in cluster]})

csv_path = os.path.join(OUT_DIR, 'near-dup.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['group_id', 'kept_file_id', 'dropped_file_id', 'cosine'])
    w.writeheader()
    w.writerows(rows)
print(f'wrote {len(rows)} drop rows · {len(groups_out)} groups → {csv_path}')

# log unreadable thumbnails (fetched OK list vs embedded) for audit
with open(os.path.join(OUT_DIR, 'unreadable.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['file_id', 'status']); w.writerows(errors)

from google.colab import files as cfiles
cfiles.download(csv_path)

## 6 · Contact sheet (first 40 groups)

Eyeball before trusting — flagged members render next to their kept original.

In [ ]:
from IPython.display import HTML

parts = ['<div style="background:#111;color:#ccc;font-family:monospace;padding:8px">']
for g in groups_out[:40]:
    imgs = ''.join(
        f'<div style="text-align:center;margin:4px"><img src="https://lh3.googleusercontent.com/d/{fid}=w300" width="200"/><br>'
        f'<span style="font-size:9px">{html.escape(by_id[fid]["name"])}</span></div>'
        for fid in [g['kept']] + [i for i in g['members'] if i != g['kept']][:3])
    parts.append(f'<div style="display:flex;gap:4px;border:1px solid #333;padding:6px;margin:6px">'
                 f'<b style="min-width:90px">{g["id"]}</b>{imgs}</div>')
parts.append('</div>')
HTML(''.join(parts))